# VideoMAE V2 · Hành động và tiếp xúc giữa hai người

Mở notebook trong Colab GPU. Chạy các ô theo thứ tự. Ô cuối cho phép thử một clip KO. Trọng số tốt nhất và ngưỡng được lưu trên Drive.

Mỗi video bổ sung nên có **một cặp người tương tác**. Với cảnh đông người, cắt vùng quanh cặp cần đánh giá trước khi gắn nhãn. Giữ cùng `source_id` cho mọi bản cắt của cùng video. Pose được trích tự động từ ảnh. Trong cửa sổ bị che, nếu nhánh pose mất dấu hoặc không phân giải được hai người, hệ thống có thể chuyển cảnh nghi vấn sang xem lại.

### Dữ liệu cần chuẩn bị

Bản này có sẵn 17 sự kiện đã xem trên RLVS, bảng nguồn và ba split. Ô cấu hình sẽ tìm dữ liệu trong /content/data hoặc Drive theo bố cục V1. Các CSV được nhúng trong notebook. Muốn dùng bảng nhãn đã chỉnh, đặt LABEL_DIR đến thư mục chứa chúng trên Drive và giữ các nguồn cùng split. Các đoạn KO ở ngoài tập huấn luyện.

Mỗi dòng events.csv mô tả **một sự kiện**. Xem clip trước và sau rồi điền vùng thời gian [start_s, end_s], thời điểm tương tác event_s và cửa sổ ngắn nhất còn đủ bằng chứng min_window_s. Với cú đánh chọn thời điểm ra đòn hoặc chạm. Với vung tay hụt chọn lúc tay đi gần người kia nhất. Quy ước contact là **tiếp xúc của tay người này với người kia tại sự kiện được xét**.

| Ví dụ | scenario | action | contact |
|---|---|---:|---|
| Một cú đánh trúng, có hoặc không có người ngã | fast_strike | 1 | touch |
| Vung tay đi ngang che người, xác nhận không chạm | arm_overlap | 0 | no_touch |
| Vỗ vai hoặc đập tay thân thiện | friendly_touch | 0 | touch |
| Có động tác giống ra đòn, điểm chạm bị che, hành động chưa rõ | occluded | để trống | unclear |
| Động tác tấn công rõ, điểm chạm bị che | occluded | 1 | unclear |

Chỉ dùng nhãn `no_touch` khi quan sát được đủ ngữ cảnh. Nhãn trống bỏ qua loss tương ứng. `unclear` là lớp có giám sát dành cho tiếp xúc bị che. Clip RLVS chỉ có nhãn hành động. Khi bổ sung nhãn cho clip đã có trong RLVS, dùng đúng split của nguồn đó.

Ví dụ một dòng minh họa: `extra/wave.mp4,scene_012,train,arm_overlap,0,no_touch,0,3,1.4,1`. Đặt `min_window_s=1` khi nửa giây quá ngắn để hiểu tình huống. Chọn các **nguồn độc lập** cho train, val, test của từng trường hợp. Ba file KO đã xem thuộc tập chẩn đoán riêng.

In [ ]:
%pip install -q 'transformers>=4.45,<5' opencv-python-headless scikit-learn

In [ ]:
from pathlib import Path
import json, gzip, base64
PREPARED = Path('/content/train_v2_inputs_9b39274d09')
# Bản chụp các CSV đã kiểm tra, giải nén vào ổ cục bộ của Colab.
packed = 'H4sIAAAAAAAC/62dS29cR5JG/4qgVRsg6JvvjF7PcuDFLLwagChLJanQNCmQJRmNwfz3qSKrKGlacrvOuRupLOPLzJsRkY9zI/P+z+vt5+3d/vH6zePn139/9frjZv/h6vH+08Ob7c3u7dXjx9vd/urxzfZu87C7v9q82e/u767e3N/tDz+vHvebh/3N49X27u3xz2NBh79/393d/LG7e3v/x83jf9+93ew3j9v9z/+13dy++s/du+2rX3f3t9u7N9tX/3H6X7/c353/7edffr1J179/rFcP9/e/37x/2G73u7v3V/uHze7u6t3D7lDV7T9v9vef3ny4Wq6e/07XvV3l614Of5RxtVw3Um3tT/W+u7/f/7a5vb152L69+e320/ZU9+bh95v7z9uH283HQ8V396c25Os0rsp1fWrAoSULeuKl/mndP3juQ93Hqvvhj6XQunN97u93u9vfb95+2u5PdX7YHIz6fvu4//Sw/fqBl+txfNZ5qG/Wv1zpS42/3pTlucLdwcMOz1nLzR8fdvttXs4Pu3k8eNH+YfeP7VV6sXCuh0pbO/yK+pct/FWt58e8273/sH/q3zebh+/W+Onuze328P+W62OV/VhlSqTK8XXPPj3jzeOn3f7xT2vN18s82DPPw69RqS+PeHHmtze/bx7+sd3ffLh/3F993tz+yJuWY835Oo42bbTimZ8rPtV5fLyPh75+/LB7eK78B2F06ONj9NYnA2fqy7Msf177j5w6HQePcp3y0eKZeHXLz8a+/7R/e3//cPPu+O/P3f0D73oKo2MEdeJc81TfoeDt86N+ev/+9rnG9w+bj4dx+zBsfomf8TRK9EOVOahx2zg79Pb27VMMPXn11f7QoVe/3b/95/cHyKM/H8Lo4NQZ+/NycqtPj7s3N7u7/fbh8277x3PNPzZqHCodT07diU3T8k0f3797dxizHp4r/b5VD258fODjYBxozEj9uco/jrPAsx1P/fs9o8Zx6mnHruUR20b7sVG/jdYvTznysWePDlxOPfv66tXr07Lh/cP9p48/WFCstB5YdZJFhc2nsp5+IX04fV7O+sxm/dM4/fwTlVDmt+sGVkisUEhd1igkrVFIXqOQskYhdY1C2hqF9DUKGWsUsobH1jU8tr0EX2P6NTy+reHxXT7JkOPYfBmIDz9ZCfGlhEAlxKJL+NINMVgJU5ZQSpM9WXo9l3D4iUoY05YwX3ry8JOVEF9KQNYsX/wBl5B0Cdn2QxRZwos31DVxDCprrFjWXLGsWK+spnr7PKGs0pLvbxPZvq9+b9/HimrrFdXXK2qsV9Rcr6hYrai0rFbUaUz7/1CLlVVWLKuuWFZbsay+YlljxbLmimXFamXNZVmvrPS39NN5RH76D1bKCkWkFR8qr1hWWbGsumJZbcWy+opljRXLim+8M7BvxQqF5K/bknEpKxTRzi353vsDWOTK5fWVyxsrlzdXLi/WN0msW1752n0L9b2yQhHL6n31J6/TWHlp/SamlcvL6zcxr1xeWb+JZeXy6vpNrCuXt/LQXFYemsvKQ3NZeWguK4+jdeWxpn1ZEbTMSii2hFiuN593V88/WQlJlhDlZY12+IlKqIst4QtKPPxkJdRLS/i37/EvL+W5DU+/Lhe3F/G4XDyMeBpxiGc+bX9YzX35l0SINZIpLi8jr1BGWaGM8GXEix/F5QaJycXfzb67uJDTgPz063JxFeJ5jt7Dr4vFJ2yJxD9Mlry4oHwOxvIX8z2+EQcX19MS/enXxeI6uLgt9dIkl6/VpxyfbzL6Li8kVijklO0jC0m+kF7PIXj4dbm4cvH4Ot3oq2Tay8uJdcr5OulIlZNWKievUs4P8kovLyavUsylK4i/ltx4Slc8nm14/Pnh9vPjzTHb8ZuMxdvNb9tblmb4VCM8BxBCnI22C/Fpaw7F5pGrqbkZSzXTYaf1IBOf3q5D8RDiaXo7TLND9HZekhFPIU6iw3IW8ZyLEpveNoGRu2l2FyNJHsbOJjByCHFZFiMWHVZSMmJTcxZjWCli6DXzXKmm1c3UbIKqdNPsYZo9jda02vhmmL4O45shml0XYahqFp41CTNXMxKcsQQUm2ZXYedqVq21NSHuRmyGgmom5zrFgqSakGxmcm6L8O1mQvJMm6DYNNusWls1zTazczMh2ZoYt1sXvt2GabbZhZ6pDBSLwaBNU7OZ3FsYDzOTe1/EM3ezzO+pGLHw7Z6rEZveNhVX88jVtNosDHo3PmJ27X2YDpum2VOMQj1Ebw+zqhhmoT8W1exuxMOIpxEL9xzmXcAwa7hh1nDDvAwYBrAMs/ceZvQchrCYUWiYJZyh+WOYvp7GyuZVwDSD0DQBOWs2YmGq2cQINg3+m+M5gxGKpxDPbMSm5jDPfErYhOLBxXGY2oU4hDiJkIycjFgMnmFoQ1TTbDMYhJmpoi9GLGaMGGLGiGm0xkcuWql/kzj73NGJ5ipfLky0xhOCv1x42mRfLiy0qaf14eXCE3K/XNhoUxvt1U5r7NRzTiuERPOagXBQIXWAQc0x6TMGtWPAZ8wLtGNOVNehMMNO5TrocJkOHKeTv0BHW9ppjYOaf1BHnZUK6TMGfcaAMQXdpizQ/CXRGun0X+j0f87BuVxIK6xU2BIVwtAoHQ6pZdBOpeansVjoxFgCNvWc3nK5kK6pa4KdU+lcTOepWmmFNKhqy1QIx8baaVMHbeqgTaVRVWlU1YAjTqPLzbbAwbHR5V+jq7FzRgoQ4hpp59B4bHSSa3SSax0O5I3GI92MN7pSxV1Do7HTOa7TOa7ThWPPmQphr3a6/euVNpUG1TkT43IhhTGd7v/6pJ1DJ7lOd2PQ/OeMh8uFKVEhdPFBd2ODzlSDLjlHhQPHoLExKBuhy7gxqDVoTA26/jvfhHqxcNL131zgHHe+fxIIYVSdL8i7XEhZxaQzzqz0GelUNTsWwnCcdP03Kf6fdKqadKqCVgwaVEF3Y0Hpf9D1Hx2potCWUjgSdIqLRptKgzHoZiwGDI2YCxXSpgZt6pd3cf9yY8TxI34rXRlhboxQ102oms19E4u5AcEkMSeTP5hMEnMySczJJDEndSeJOQGS1KUk5vhIMncvJJPRlszBlZSMkyTlJOruGeNhJgUwZXXtjfGwbJzEnJNM2djZnG5SEVmMj5jT2slkeqZizFzUvUrGR8wJdTV2mnO4yaTVJpOgn8yh/FSNj5hs4FSVmc1QYB65GTM3Y+ZmzNzUdWOqw4yPmOMjyZxPTybdPJljpcmceknmEG8y12Skrq6VM07SjZOYis01BGkYMw91DZ/xEXMBQjLHDdNQZjZjgTlunaaxszmsmMyNEcmcPUlT3dVo7DyNncPEs7liI4W6n9LY2VzukczppmRuBjGXLS5Gq8Sq1cWIzS2Pi7rBdBix8RBzdaqhjjmpmo2PmI1NNuwvm2VUTuqiWuMjprOVgxl6l7O6m9f4iDm0nA03zIYbqs426C8b9JeLarYxs6F3uShLmWg2YWHQXzboLxv0lw36y1V1mHEwc99NNtAxG+iYDXRU4665pjEbYpkNscyGWGZDLLOBjtlAx2ygo5peDbDM5rMO2TDHbJhj7qrDjI8Y5qgWQoY5ZnP1aTbMMRvmmA1zzEP1tolm9V0GY2bDK7NBjtkgx2yQYzbIMU9lKuMjSmt8xNxklw3rzAZXZvU5HUMczQcOTLJiMdSwLKrZ6vMuzYi7EZvPb6hPw5gv2pj8pGKyXoqhhsVQw2KooUmMKuYdQDHgr2TVbOMjBo8Us3stZiFUzAypvphkwF9R3qk+1mTAXzE5f6Woj1sZHzHwrhh4Vwy8K1V90Et90cs4SVWmMuFsWm0QWjEIrRiEVpp6ZhPOTX31zfiIQWhqBDMIrRiEVrr6Sp7xEQOyirlWV00XhoIVQ8GKSdsrBmQVkzxXhupt4yMGoRVDsorJvCsGg5Wpvh5pnMQwtDLV9yNNSBoaVdRnaA2NKoZGlVBf+jTfFTSf2zRaJTaf6jQYrBoYVU3+WzUkq5rkObMnqkmJjaWS+has+UaowWDVpLCZDWg1GKwaDFazarbxEUOyqnlBX7P6XrDxEdNfhmTVomo20WxYQTWbyGpS2AzUqWpuViO+a7axlEkkqyaRrBoKph7ZsKhqWFRt6uPfxkcMi6pNWcpEs3FOw6KqOX9azfnTakBW7arDjI8YClYNBVNjkKFg1YCsanLBqqFg1VCwaliUGvBNOlc1EKwaCFYNBKuGY1WTzlWnMpXxEaU1PmIAWjUArRqAVg1AqwagVQPQxBOb15DN5JE1A9CaAWjN5JE1Q9+aAWjNADSzSG8mj6wZgNYMQGsGoDUD0JrJI2vm9Gkz6K4ZgtYMQWuGoDWTSNYMQWuGoDVD0MzOt5lEslZUzcZShqA1Q9CayQxoBr8ZRtEMQWsG4zfDd1tVz2x8xGyomllpm5yqZghaU+OI8k73zCaaDbtTFRuC1gxBa4agNUPQmkkFa11ZykSzIWjNgKxmQFYzhxqbAVnNgCzDw5sBWc2ArGZAVjMJWc3kVDUDstpUpjI+YmBUMzCqhYlIQ7KaIVnN8CThX2aS64sSJyPO15vPOyqupmbV2cOIjYcYGNUNUOrmRrJugFI3QKkbrNMN1jGLv24ysrrhST2rZhtLGSTUzV0a3fAkc9yjGyTUTUZWL6rZJpoNT+qGJ/WizGyGApOS1Q0T6oYJdcOEumFC3aRDmL1vN+/lunnb0w1Q6gYodUNXelOWMhFpUrJMhlFXM7uaLwwT6iouXIcZH1FaE83meGA3MKobntRNVlUfqrdNNBuSZZhjNzypm8SobmBUN1lV3cCobnhSV1oTzYZFdXPPVTcsqpvcpq5YlLkkSzR6GJ40THLTMMlNZrIYSYmrETcj7kZsfMRAHTMzD4NWRlY1G+c0qTrDoJWRlaXMEGT6y3CZYe6OGwatDINWhkErw6TqmNXuMGRlmFSdYbDMMFhmVNVhylImmk2qjtnUDMN0hmE6w2CZ0dQzm2g2TGcYpjPMa8xhcnWGgTrDQJ1huMwweGR0ZSoTzgbMDLWCU+sCNVWpMcxkCQ0DZpzWmNlkCQ1DdYYBM8NkCQ0DZsZUpjLhLLSizca7psEM02CGadJWpklbmSZtZS7KUsZD0t/ST0ZutCHrNs+dXd3GR03+yTSQZGbV7GLEJi6Ls5Vpt2l2VRWrmpsRm1HQbGKn2cROsxGdZiM6zZmRafaS0+wlp3nFb9Zh0+xD52E3iJNaZ69G3Iy4G/Ew4inExr9GMhVnIzY+MoyPDOMjw5j5sE3gYjNoHzYKomJjqWksNY2lponmaSxlxuwwMRXGUmEsFcZSYUa/EJYShjKpGLGIUTfMiZFYihE3I+5GLDzE5L1EWkzFquZsxNWIuxEPI55GbKLZbPfD7NjD7NjD7NjDnFUJkxMRJq3BnIoMk9YQJq0himq2MbPJiQiT1hDmuEmY4yZhkiLC5DWE4UlheFIYnhSGJ4XhSWF4UhieFIYnhTmsEiaxIQyMCgOjwiQ2hCFZYRIbwiQ2hElsCHNaJUxWRJisiDBZESZdO0xSRJgE5DB5DWHyGsLkNcRQvW3MbF49h3kjGea1Xpj3TGFeNIXBtDGVqczIq7TGR8yJk1CbBHNcJcw9vqEGT3PWJS6y84vy15vnQE4Xy5bTjStAmamwUGGlwkaFnQoHFU4qDChM1HUSdYBEHSBRB0jUARJ1gETNQTs10zjO1P4ZN5VaI1NrZBqOmYZjpvYv1ByFmqPQcCw0HAt1gEIdoFAHoI9YaThWav9K7V+pNSq1RsXWoOFYaTg2ao5GzdFoODbqq406QKMO0Kg5aIWd9k2nfdNpU2lsDDriDOqpg3rqoNYY1BoDW4N6Kh2pJjXjpCPVpNaY1BqTWmPS4JjUHNT8Qa0RNBqDRmNQ+wc0I3xCusPJCxYmKqxU2KiQ2oKSikxJRU7UHJRUZLqpzpRUZLpvyJRU5ITNAYc4uv3POI4pqciZWoOSikxJRaakIlNSkQs1ByUVmZKKTElFpqQiU+CQCzYHDUf6iJRUZEoqcsVNpeFISUWmpCJTUpEbNQclFZmSikxJRaakIlNSkSmpyA3bkcYxbWmn9u/U/hSq5I6fkYZjp2bs1IwUx2SKYzLFMZnimExxTKY4Jg9qR6yjZqQ4Jk9qf0pVMqUqeeJepVGFdTQ2KBzJQWMDU5WgZgzYq3TnUBYsTFSYqbBQYaPCToWTCqn9qTUoASoJ11ipkJqRvqoqlMfQXXWhPKZQHlMybiq1Bl1VF8pjCl1xFDrHUThS8DCOPY7imEITRwrlOIUmjhQKgArlMYXymEIzR0qlLkd5TKE8plRsDhqO1FUpjikUxxSKY0rDz0jt36j9aYWdRiPFMYXimEJxTKE5LqVja9BopDgGD+MU4xSKcQqlMYXSmEJpTBm4V6kZKY4pFMcUimPKpMMx5TiFcpxCOU6hHKfQ9JhC02MKJUCFEqBCCVChBKhQkAOfkK5U64KFmQoLFTYq7FQ4qJBaP1EdFlIz0nycSmlMpem4lebjVMp/6C6uZiyk1qC5CpXyn0r5T6X8p1L+UymOqRTHVspxKkVOlQKgSvdxlW4AasF2pHFM+wZPxxTjVJqPUyn/qZT/VMp/KuU/lSbyVJrIUykAqhQAVQqAKk2rqTStBnscBUCVAqBKAVClHKd23Dk0HCkAqhQA4YGDAqBK03EqBUCVAqBKs2oq5Th4NKb8p1L+Uyn/qfRYVaX8p1L+Uyn/wXMjxT+V4p9K8U+l+KfSdJwa2BrQHHDYoAucRrN4GqU4jZ7Hagvum06F0G3oqrHRZJyWcI2ZCqn9aRZPo9yoUW5E3/62jIX0ETN+xEmFdICjfkORSiu4RjpSUTLSKBmhm8ZGOUWjoLpRNNooGWl0u9kqtgYNKrqIo3v/hoc4HMbYVfkzUvvTBBdcIeUbjfKNRvlG69T+FIy0jnsVm5FGI61wUDNSvtEo32iUbzSa4NIoGGkUjDQKOBoFHI0Cjkavf2kUcDQKOBoFHI0muLSgdqSEo1HC0SjhaDTBpVHC0SjhgA5H58a+YGGiwkKFjQpxpw4qnFRI3YYijk4RR6eIo9MTTp0ijk4RR6cZLnTh2OkJp04zXDrNcOk0w6VTjNMpxqGr8U4TXDqlMZ0muPSCn5Hanx4b6TRPpdM8FXrcoFOM0ymN6RU3lVqD8p9O+U+n/KdT/kM31Z3yn075T6f8p1Oq0ul2vNPEGEoqOl7jUv7T8TSORxzeOdSMHZuRRiNtKT2n1CnG6RTjdIpx+sCdQ81I+Q/2G8qNOj2m1CnF6ZTi9Ik7h1oD66g1KMTpFOJ0CnE6PaXUaWJMp9iIpvCNBQsTFWYqLFTYqXBQITYjDEf6imNQijMoxRmU4gya4TIo/hkU/9AFx6AUZ9B7akbGTaXWoBRnZGwNGo00GYe+NxyUGw3KjQblRqPgZ6T2p/hnUPxDl/+Dpv8Myo0GTf8ZFT8jjX/KjQblRoPiH7qLGzRteFBuNCg3GhT/DIp/BsU/g56LonvqQbnRoNxoUG40KDcaHXcOtT/d/w+8x8GLY7yOwwsA7KuU/4yBzUHDkVZI+c+g/GfQ802D5g0NSpwGJU6DEidsfnowatC0oUH5z6D8Z1D+MzD/oWlDNKjmgoWJCgsVViocVDipkJqRdio9GjUp/5mU/0zKfyblP5PyH5pvOin/mTSLZ9Isnkn5z6Q0ZlIaQ6f/SWnMpDRmFtxUag1KYyY9UzUpxqGLsUlpzKQ0ZtIsnlnxM1IzUqgyaTIOXRpPClUmhSqT5tTMhp+RmpHSmNmwGeloTB+RspFJszgnhSqTspFJ2cikr/HptnFSGDMpGpmUxk+KcSeFMZPCGLoXn3gXh/cNeImLV2N44MA+zs1BoxHrqP1pMs6kyTiTwphJYcykMGZSGEOtEQsWJirMVFiosFLhoEJqRspGgp6pCpqNE5TGBKUxQdlIJGwOODpSbhyUxgSlMUFpTFAaEzQbJ2g2TlD+ExTHBMUxQZNjgnKcoBwnKI6Jgs1Bw5H6OM1xCYpjouKmUjNSjhM0x4Uux4LimKC3uAXlOEGTY4ICoKA4JiiOCYpjgia5BOUxQXlMUB4T9BxfUJATlMfQvUpQHhM0OSZockxQkBOUx8TAvUrDkWZHBAUyQYFMUCAT9IVM0FcAQUlOULASWEfDkWbHBN7l4PUYBTKBB7kvQOb11avXjx9vd/vHnx9uPz/efN7cXr95/Pz6769ef9zsP1zdbn7b3v6l8n+5v3up4pdfb9JygjALUiehPW3+mPi042Di06qTiZU2hHgarTGyaXQIO+XTRUZMnEzNeTFi0WG5iKjIzXRYF8GcT6sjKBYxlaepWbnnaVyHYhHOZTFa0eqSshGbR87CR0pJRizCuZip5vz5ZyY24Vy6afYwzTYz1fmrp0hcl2TEYuStaTFiEc/VrMGqGYbOX29j4pqNWMRzbSIwaheBUYdxz2F6exonCWOqEKZqJiSb2dm0bLSm1Waea0UMns3Mc62ZZjfTbBORbRitEU/TahNSIax8vs+WiZPRir4+3xfKxCYeu5khzzcyQrEIqfPNfEzcFyM2zTaL1m4Wredbt5jYbGC7CechHnkYNjMMjxpmej1fHgLFwjvPt08wsVmPDDNfnI/pM7FZ8A4zv56PMzNxmMAwq+VhxoIRxlShOsyMQobBDcPghtmbTPPSZZqF1DQbm2nw30yq2dWImxF3Ix5GLNxz5va39JPSG7Hps2z6LKs+C9lnZkgoi6vc0LhZkqzcjColy8rNqFSKrNyMaqXKys2oWEyEFxPhZmc7zTp6mnc554PVUGwixFjKrP6neQM1zdvoaYjP8Xjo5vMOikcX4pmMeAhxiGYfT2IJsWh2mFWh2S2FWSmEGf7MTststMIEs9mkhXmbHCaz5JyXysRm2D0nNi6XJv3RQ3j0DF6iB9sSPfWVsA5mJyd6roXewJToeYhEL0RINK090VzxRBOwE01OTvTsTqIJuImeiM70uHCmF6JlGsaZ3oie6f1UmR76y/REXKanxTI9LZTpoR96zDTTOM70PESmOf+ZRlWhZ+kLPWdeaFQVOjkWeni30POphR7eLPRIZKGn/gqNqkKnx0JPRBV6WgibkU5ylV6mUelFE5VewlDpDQWVzlWVzlWVzlU0Giu9S63SoKo0qCo92FjpXFXpmrPiD6LTu20ajapGo6olXCPs1UbjuNHv/jR600ijt3A0ulttdARodD5udD5u9IR6o8e3G13mNjp0NHpCteGP4i34S/NUh79QTmvM+NPW+CvM9PuNDQvppybxZ5hpVPWBvxhKHWDij1vSGunZ9rHgD/jRD3gk/AE3/JEy+owFfxaJfhaj4U//0M6hl5sM/EEtGlZj4q+U0Pst6Ycf8McN8Jcf8D38tMaMP4uAL6mnNVLqMCu++hvfUk3vfu34emPqAPRWnInv/qez48Q3v9BAxhe/JHz1I62RwqMo+D49ep1OxRex4SvDqA5fboXvb6I6fO/PS0tf/+//AaVgCsBIAAIA'
for name, body in json.loads(gzip.decompress(base64.b64decode(packed))).items():
    path=PREPARED/name
    path.parent.mkdir(exist_ok=True,parents=True)
    path.write_text(body,encoding='utf-8')

from pathlib import Path, PurePosixPath
import base64, gzip, json, shutil, zipfile
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ROOT = Path('/content/data')
# Các bảng đóng gói nằm trong ô ngay trước. Có thể sửa LABEL_DIR để đọc nhãn riêng.
LABEL_DIR = PREPARED
SPLITS = PREPARED / 'splits'
OUT = MY_DRIVE / 'train_v2_contact_prepared_run1'
OLD_BEST = None
EPOCHS, BATCH, ACCUM, SEED = 5, 4, 4, 42
BASE = 'MCG-NJU/videomae-base-finetuned-kinetics'
RLVS = DATA_ROOT/'dataset'/'Real Life Violence Dataset'

# Chỉ lấy đúng video có tên và thư mục lớp trong split. Bỏ qua các bộ khác trong zip.
splits = pd.concat([pd.read_csv(SPLITS/f'rlvs_{s}.csv') for s in ('train','val','test')])
required = {tuple(PurePosixPath(p).parts[-2:]):DATA_ROOT/p for p in splits.path}
missing = {key:path for key,path in required.items() if not path.is_file()}
if missing:
    candidates = [MY_DRIVE/'dataset'/'Real Life Violence Dataset',
                  MY_DRIVE/'Real Life Violence Dataset']
    project = MY_DRIVE/'lab_ATvsBMTT'
    if project.is_dir():
        candidates.extend(p for p in project.rglob('Real Life Violence Dataset') if p.is_dir())
    for folder in candidates:
        for key,dest in list(missing.items()):
            src = folder.joinpath(*key)
            if src.is_file():
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copyfile(src,dest)
                missing.pop(key)
        if not missing: break
    if missing:
        archives = sorted(project.rglob('dataset-*.zip')) if project.is_dir() else []
        for archive in archives:
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    parts = PurePosixPath(member.filename.replace(chr(92),'/')).parts
                    if len(parts)<3 or parts[-3].casefold()!='real life violence dataset': continue
                    key = tuple(parts[-2:])
                    if key not in missing: continue
                    dest = missing[key]; dest.parent.mkdir(parents=True,exist_ok=True)
                    tmp = dest.with_suffix('.part')
                    with z.open(member) as src, tmp.open('wb') as target: shutil.copyfileobj(src,target)
                    tmp.replace(dest); missing.pop(key)
            if not missing: break
    if missing:
        raise FileNotFoundError(f'Còn thiếu {len(missing)} video. Giải nén bộ RLVS vào {RLVS}. Ví dụ: {next(iter(missing.values()))}')
print(f'Đã có {len(required)} video RLVS và {len(pd.read_csv(LABEL_DIR/"events.csv"))} sự kiện.')

### Chia dữ liệu trước khi cắt cửa sổ

Notebook giữ nguồn, file trùng byte và các sự kiện cùng clip trong cùng một tập. Các bản chỉnh sửa khác byte cần được khai báo qua source_groups.csv. Bảng kiểm tra bên dưới đếm theo nguồn để bạn nhìn thấy nhóm còn thiếu.

In [ ]:
import hashlib, json, math, random, os, warnings
from functools import lru_cache
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix

CONTACT = ['no_touch', 'touch', 'unclear']
SCALES = (.5, 1., 3.)
MEAN = np.array([.485,.456,.406], np.float32)
STD = np.array([.229,.224,.225], np.float32)
def filepath(raw):
    return str((DATA_ROOT / str(raw).replace(chr(92), '/')).resolve())

def file_hash(path):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

@lru_cache(None)
def info(path):
    cap = cv2.VideoCapture(path)
    try:
        n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
        if n < 1 or not np.isfinite(fps) or fps <= 0:
            raise ValueError(f'Không đọc được video {path}')
        return n, fps
    finally:
        cap.release()

def make_manifest():
    gp = LABEL_DIR/'source_groups.csv'
    groups = {}
    if gp.exists():
        for row in pd.read_csv(gp,dtype=str,keep_default_na=False).itertuples():
            path, source = filepath(row.path), row.source_id.strip()
            if not source or groups.setdefault(path,source) != source:
                raise ValueError(f'Kiểm tra source_id: {path}')
    extras = []
    ep = LABEL_DIR/'events.csv'
    for row in (pd.read_csv(ep,keep_default_na=False,dtype=str).to_dict('records') if ep.exists() else []):
        path, source = filepath(row['path']), row['source_id'].strip()
        if not source or groups.setdefault(path,source) != source:
            raise ValueError(f'Kiểm tra source_id: {path}')
        split, scenario = row['split'].strip(), row['scenario'].strip()
        action = {'':-1,'0':0,'1':1}[row['action'].strip()]
        contact = CONTACT.index(row['contact'].strip()) if row['contact'].strip() else -1
        if split not in ('train','val','test') or (action < 0 and contact < 0):
            raise ValueError(f'Split hoặc nhãn không hợp lệ: {path}')
        if scenario in ('arm_overlap','friendly_touch') and action != 0 or scenario=='fast_strike' and action!=1:
            raise ValueError(f'Nhãn hành động không phù hợp với {scenario}: {path}')
        n, fps = info(path)
        a,b,t = (float(row[k]) for k in ('start_s','end_s','event_s'))
        m = float(row['min_window_s'] or 0)
        if not (0 <= a <= t < b <= n/fps+1e-6 and 0 <= m <= min(3,b-a)+1e-6):
            raise ValueError(f'Mốc thời gian sai: {path}')
        extras.append(dict(path=path,source=source,split=split,scenario=scenario,action=action,contact=contact,a=a,b=b,t=t,minimum=m))
    files = [SPLITS/f'rlvs_{s}.csv' for s in ('train','val','test')]
    old = all(p.exists() for p in files)
    if any(p.exists() for p in files) and not old:
        raise ValueError('Bộ CSV V1 thiếu split')
    if old:
        base = pd.concat([pd.read_csv(p).assign(split=s) for p,s in zip(files,('train','val','test'))],ignore_index=True)
        base['path'] = base.path.map(filepath)
    else:
        base = pd.DataFrame([(str(p.resolve()),y) for folder,y in [('NonViolence',0),('Violence',1)] for p in sorted((RLVS/folder).rglob('*')) if p.suffix.lower() in {'.mp4','.avi','.mov','.mkv'}],columns=['path','label'])
    if base.empty or not base.label.isin([0,1]).all():
        raise ValueError('Kiểm tra RLVS và nhãn 0/1')
    base['source'] = [groups.get(p,p) for p in base.path]
    signatures = {p:file_hash(p) for p in set(base.path)|{r['path'] for r in extras}}
    if not old:
        # Link both declared sources and exact duplicates before choosing folds.
        parent = {}
        def root(k):
            while k in parent:
                k=parent[k]
            return k
        for r in list(base[['source','path']].to_dict('records'))+extras:
            left,right=root(('s',r['source'])),root(('h',signatures[r['path']]))
            if left!=right: parent[right]=left
        keys = [str(root(('s',s))) for s in base.source]
        folds = StratifiedGroupKFold(7,shuffle=True,random_state=SEED)
        base['split']='train'
        for fold,(_,ids) in enumerate(folds.split(base,base.label,keys)):
            if fold<2: base.loc[ids,'split']=['val','test'][fold]
        explicit={}
        for r in extras:
            k=str(root(('s',r['source'])))
            if explicit.setdefault(k,r['split'])!=r['split']: raise ValueError('Nguồn phụ trùng split')
        base['split']=[explicit.get(k,s) for k,s in zip(keys,base.split)]
    rows=[]
    for r in base.itertuples():
        n,fps=info(r.path)
        rows.append(dict(path=r.path,source=r.source,split=r.split,scenario='base',action=int(r.label),contact=-1,a=0.,b=n/fps,t=0.,minimum=0.))
    df=pd.DataFrame(rows+extras)
    df['digest']=df.path.map(signatures)
    for key in ['source','path','digest']:
        if df.groupby(key).split.nunique().max()>1:
            raise ValueError(f'Rò rỉ dữ liệu giữa các split theo {key}')
    return df

def windows(df):
    rows=[]
    for r in df.to_dict('records'):
        span=r['b']-r['a']
        lengths=[span] if r['scenario']=='base' else sorted({min(w,span) for w in SCALES if min(w,span)+1e-7>=r['minimum']})
        for length in lengths:
            rows.append(dict(r,length=length))
    return pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
manifest = make_manifest()
samples = windows(manifest)
display(manifest.groupby(['split','scenario','action','contact']).source.nunique().rename('số nguồn').to_frame())
for split in ['train','val','test']:
    if set(manifest.loc[(manifest.split==split)&(manifest.action>=0),'action'])!={0,1}:
        raise ValueError(f'{split} cần cả hai nhãn hành động')
if not (manifest.contact>=0).any():
    print('Chỉ có nhãn hành động. Có thể train baseline. Nhánh tiếp xúc chưa đủ dữ liệu để phát cảnh báo xác nhận.')
events_only=manifest[manifest.scenario!='base']
display(events_only.groupby(['scenario','split']).size().unstack(fill_value=0))
print('Số cửa sổ bổ sung',len(samples[samples.scenario!='base']))


### Trích ảnh và dấu hiệu tiếp xúc

Với clip có mốc tương tác, lấy 16 frame dày quanh mốc. Với RLVS chỉ có nhãn cả clip, lấy 16 frame rải trên toàn clip. Giữ tỉ lệ ảnh và thêm viền để cả hai người còn trong khung.

Keypoint R-CNN tìm người, vai, khuỷu và cổ tay. Trong từng cửa sổ, ghép hai người giữa các frame bằng vị trí tâm khung. Khi phát hiện quá hai người, thiếu người hoặc ghép mơ hồ, đặt mặt nạ thiếu quan sát. Nhánh tiếp xúc học từ chuyển động của khớp và đặc trưng ảnh. Khi hai người giao nhau, ID vẫn có thể đổi nên ô xem trước hiển thị kết quả ghép cặp để rà soát.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision.models.detection import keypointrcnn_resnet50_fpn, KeypointRCNN_ResNet50_FPN_Weights
from transformers import VideoMAEForVideoClassification
from tqdm.auto import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
JOINTS=[0,5,6,7,8,9,10,11,12]
PAIR_DIM=2*len(JOINTS)*3
POSE_DIR=Path('/content/contact_pose_cache')
POSE_DIR.mkdir(exist_ok=True,parents=True)

@lru_cache(1)
def detector():
    net=keypointrcnn_resnet50_fpn(weights=KeypointRCNN_ResNet50_FPN_Weights.DEFAULT,min_size=480,max_size=800)
    return net.to(device).eval()

def letterbox(rgb):
    h,w=rgb.shape[:2]
    scale=224/max(h,w)
    w2,h2=max(1,round(w*scale)),max(1,round(h*scale))
    img=np.full((224,224,3),114,np.uint8)
    x,y=(224-w2)//2,(224-h2)//2
    img[y:y+h2,x:x+w2]=cv2.resize(rgb,(w2,h2))
    return img

def indices_for(start,length,n,fps,training):
    edges=np.linspace(start*fps,(start+length)*fps,17)
    pos=np.random.uniform(edges[:-1],edges[1:]) if training else (edges[:-1]+edges[1:])/2
    return np.clip(pos.astype(int),0,n-1)

def detected_frames(images,ids,digest):
    # Cache normalized joints from original RGB, before augmentation.
    directory=POSE_DIR/digest
    directory.mkdir(exist_ok=True)
    found={}
    missing={}
    for i,img in zip(ids,images):
        i=int(i)
        file=directory/f'{i}.npy'
        if file.exists(): found[i]=np.load(file,allow_pickle=False)
        else: missing[i]=img
    keys=list(missing)
    for k in range(0,len(keys),2):
        group=keys[k:k+2]
        xs=[torch.from_numpy(missing[i]).permute(2,0,1).to(device).float()/255 for i in group]
        with torch.inference_mode(): preds=detector()(xs)
        for i,p in zip(group,preds):
            keep=(p['scores']>=.75)&(p['labels']==1)
            points=p['keypoints'][keep][:,JOINTS,:2].detach().cpu().numpy()
            scores=p['keypoints_scores'][keep][:,JOINTS].detach().cpu().numpy()
            h,w=missing[i].shape[:2]
            # Coordinates normalized in the original image. Third channel is joint availability.
            valid=scores>2
            arr=np.concatenate([points/np.array([w,h],np.float32),valid[...,None]],axis=-1).astype('float32')
            arr[~valid]=0
            np.save(directory/f'{i}.npy',arr)
            found[i]=arr
    return [found[int(i)] for i in ids]

def pair_sequence(detections):
    sequence=np.zeros((len(detections),2,len(JOINTS),3),np.float32)
    previous=None
    for j,people in enumerate(detections):
        if len(people)!=2: continue
        # Shoulders anchor identity. At least one visible wrist on one person.
        if not people[:,1:3,2].all() or not people[:,5:7,2].any(): continue
        centers=people[:,1:3,:2].mean(axis=1)
        if previous is None:
            order=np.argsort(centers[:,0])
        else:
            d1=np.linalg.norm(centers-previous,axis=1).sum()
            d2=np.linalg.norm(centers[::-1]-previous,axis=1).sum()
            if abs(d1-d2)<.025 or min(d1,d2)>.65: continue
            order=[0,1] if d1<d2 else [1,0]
        sequence[j]=people[order]
        previous=centers[order]
    return sequence

def read_window(path,start,length,digest,with_pose=True,training=False):
    n,fps=info(path)
    ids=indices_for(start,length,n,fps,training)
    cap=cv2.VideoCapture(path)
    images=[]
    try:
        for i in ids:
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(i))
            ok,bgr=cap.read()
            if not ok: raise ValueError(f'Lỗi frame {i}: {path}')
            images.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    finally: cap.release()
    pairs=pair_sequence(detected_frames(images,ids,digest)) if with_pose else np.zeros((16,2,len(JOINTS),3),np.float32)
    rgb=np.stack([letterbox(img) for img in images])
    if training and random.random()<.5:
        rgb=rgb[:,:,::-1].copy()
        pairs[...,0]=np.where(pairs[...,2]>0,1-pairs[...,0],0)
        pairs=pairs[:,:,[0,2,1,4,3,6,5,8,7],:]
    if training and random.random()<.5: pairs=pairs[:,::-1].copy()
    rgb=(rgb.astype('float32')/255-MEAN)/STD
    return torch.from_numpy(rgb).permute(0,3,1,2),torch.from_numpy(pairs.reshape(16,PAIR_DIM).copy()),ids

class Clips(Dataset):
    def __init__(self,df,training=False):
        self.df=df.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        if r.scenario=='base': start=r.a
        else:
            offset=random.uniform(.35,.65) if self.training else .5
            start=float(np.clip(r.t-offset*r.length,r.a,r.b-r.length))
        x,g,ids=read_window(r.path,start,r.length,r.digest,r.scenario!='base',self.training)
        # Do not supervise a moment of contact if temporal sampling misses it.
        _,fps=info(r.path)
        target=int(r.contact) if np.min(abs(ids/fps-r.t))<=1.1/fps else -1
        return x,g,int(r.action),target

def loader(split,training=False):
    df=samples[samples.split==split].copy()
    sampler=None
    if training:
        base_ids=np.flatnonzero(df.scenario.to_numpy()=='base')
        event_ids=np.flatnonzero(df.scenario.to_numpy()!='base')
        # Visit all base clips once. Limit annotated-window repeats on this small pilot.
        epoch_ids=np.concatenate([base_ids,np.tile(event_ids,3)]).tolist()
        sampler=SubsetRandomSampler(epoch_ids)
    # Pose inference uses the GPU in the main process.
    return DataLoader(Clips(df,training),batch_size=BATCH,sampler=sampler,num_workers=0)

In [ ]:
import matplotlib.pyplot as plt
def preview(row):
    start=float(np.clip(row.t-row.length/2,row.a,row.b-row.length))
    x,g,ids=read_window(row.path,start,row.length,row.digest)
    fig,axes=plt.subplots(4,4,figsize=(10,10))
    cap=cv2.VideoCapture(row.path)
    w,h=cap.get(cv2.CAP_PROP_FRAME_WIDTH),cap.get(cv2.CAP_PROP_FRAME_HEIGHT);cap.release()
    wh=np.maximum(1,np.round(np.array([w,h])*224/max(w,h)))
    for ax,img,people,i in zip(axes.flat,x,g.reshape(16,2,len(JOINTS),3).numpy(),ids):
        ax.imshow((img.permute(1,2,0).numpy()*STD+MEAN).clip(0,1))
        for person,color in zip(people,['lime','orange']):
            pts=person[person[:,2]>0,:2]*wh+(224-wh)//2
            ax.scatter(pts[:,0],pts[:,1],c=color,s=8)
        ax.set_title(f'frame {i}');ax.axis('off')
    plt.tight_layout();plt.show()
for scenario in ['arm_overlap','fast_strike','friendly_touch','occluded']:
    examples=samples[(samples.scenario==scenario)&(samples.split=='train')]
    if len(examples): preview(examples.iloc[0])

### Huấn luyện hai đầu ra

Đầu hành động dùng đặc trưng VideoMAE của cả đoạn. Đầu tiếp xúc kết hợp đặc trưng ảnh với chuỗi khớp của cặp người. Đánh giá cả hai thứ tự của cặp rồi lấy trung bình để thứ tự A/B không làm đổi điểm. RLVS vẫn đóng góp vào loss hành động khi thiếu nhãn tiếp xúc. Chọn best.pt theo F1 hành động và macro-F1 tiếp xúc trên validation có đủ lớp. Trường hợp thiếu nhãn sẽ được ghi rõ trong kết quả.

Đặt OUT mới cho mỗi lần train. Đặt OLD_BEST nếu muốn khởi tạo từ state_dict phân loại hai lớp của V1.

In [ ]:
class ContactVideoMAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.video=VideoMAEForVideoClassification.from_pretrained(BASE,num_labels=2,ignore_mismatched_sizes=True)
        self.pose=nn.GRU(PAIR_DIM,64,batch_first=True,bidirectional=True)
        self.contact=nn.Sequential(nn.Linear(self.video.config.hidden_size+128,128),nn.GELU(),nn.Dropout(.2),nn.Linear(128,3))
        self.action_pose=nn.Linear(128,2,bias=False)
        nn.init.zeros_(self.action_pose.weight)
    def forward(self,x,g):
        tokens=self.video.videomae(pixel_values=x).last_hidden_state
        pooled=self.video.fc_norm(tokens.mean(1)) if self.video.fc_norm is not None else tokens[:,0]
        action=self.video.classifier(pooled)
        swapped=g.reshape(len(g),16,2,-1).flip(2).reshape_as(g)
        _,h=self.pose(torch.cat([g,swapped],0))
        encoded=h.transpose(0,1).reshape(2*len(g),128)
        correction=self.action_pose(encoded.reshape(2,len(g),128).mean(0))
        action=action+correction*pose_quality(g).unsqueeze(1)
        logits=self.contact(torch.cat([pooled.repeat(2,1),encoded],1))
        return action,logits.reshape(2,len(g),3).mean(0)

def masked_loss(logits,target):
    valid=target>=0
    if valid.any(): return F.cross_entropy(logits[valid],target[valid])
    return logits.sum()*0

def pose_quality(g):
    seen=g.abs().sum(-1)>0
    return (seen.float().mean(1)>=.5)&seen[:,5:11].any(1)

@torch.inference_mode()
def predictions(net,split):
    net.eval()
    rows=[]
    for x,g,action,contact in loader(split):
        a,c=net(x.to(device),g.to(device))
        for av,cv,y,z,q in zip(a.float().softmax(-1)[:,1].tolist(),c.float().softmax(-1).tolist(),action.tolist(),contact.tolist(),pose_quality(g).tolist()):
            rows.append([av,*cv,y,z,q])
    df=samples[samples.split==split].reset_index(drop=True).copy()
    df[['p_action','p_no_touch','p_touch','p_unclear','y','z','pose_ok']]=pd.DataFrame(rows)
    return df

def head_scores(df):
    actions=df[df.y>=0]
    af=float(f1_score(actions.y,actions.p_action>=.5,zero_division=0))
    labeled=df[df.z>=0]
    cf=None
    if set(labeled.z)=={0,1,2}:
        pred=labeled[['p_no_touch','p_touch','p_unclear']].to_numpy().argmax(1)
        cf=float(f1_score(labeled.z,pred,labels=[0,1,2],average='macro',zero_division=0))
    return af,cf

def atomic_save(obj,path):
    tmp=path.with_suffix('.tmp')
    torch.save(obj,tmp);os.replace(tmp,path)

In [ ]:
if device.type!='cuda': raise RuntimeError('Chọn GPU trong Colab trước khi train.')
if EPOCHS<1 or ACCUM<1: raise ValueError('EPOCHS và ACCUM phải dương')
OUT.mkdir(exist_ok=True,parents=True)
if (OUT/'best.pt').exists(): raise FileExistsError('Chọn OUT mới để giữ best.pt hiện có')
samples.to_csv(OUT/'manifest.csv',index=False)
net=ContactVideoMAE().to(device)
if OLD_BEST:
    net.video.load_state_dict(torch.load(OLD_BEST,map_location='cpu',weights_only=True))
# Check actual pair visibility on the labeled windows using the Colab runtime.
# This also fills the pose cache used by the training loader.
event_windows=samples[samples.scenario!='base'].reset_index(drop=True)
audit=[]
net.eval()
with torch.inference_mode():
    for j in tqdm(range(len(event_windows)),desc='Kiểm tra pose trước train'):
        row=event_windows.iloc[j]
        x,g,y,z=Clips(event_windows)[j]
        audit.append(dict(path=row.path,split=row.split,scenario=row.scenario,
                          length=row.length,target=z,pose_ok=pose_quality(g[None]).item()))
        if j==0:
            before,contact_logits=net(x[None].to(device),g[None].to(device))
            assert before.shape==(1,2) and contact_logits.shape==(1,3)
            assert torch.isfinite(before).all() and torch.isfinite(contact_logits).all()
pose_report=pd.DataFrame(audit)
if len(pose_report):
    pose_report.to_csv(OUT/'pose_preflight.csv',index=False)
    display(pose_report.groupby(['split','scenario']).pose_ok.agg(['sum','count']))

train=loader('train',True)
body=list(net.video.videomae.parameters())
body_ids={id(p) for p in body}
heads=[p for p in net.parameters() if id(p) not in body_ids]
opt=torch.optim.AdamW([{'params':body,'lr':2e-5},{'params':heads,'lr':2e-4}],weight_decay=.05)
steps=math.ceil(len(train)/ACCUM)*EPOCHS
warm=max(1,round(.1*steps))
def rate(i):
    if i<warm: return (i+1)/warm
    return .5*(1+math.cos(math.pi*(i-warm)/max(1,steps-warm)))
schedule=torch.optim.lr_scheduler.LambdaLR(opt,rate)
scaler=torch.amp.GradScaler('cuda')
best=-1.; history=[]
seen_contact=set()
for epoch in range(EPOCHS):
    net.train();opt.zero_grad(set_to_none=True)
    for i,(x,g,y,z) in enumerate(tqdm(train,desc=f'Epoch {epoch+1}')):
        seen_contact.update(z[z>=0].tolist())
        x,g,y,z=[v.to(device) for v in (x,g,y,z)]
        first=i//ACCUM*ACCUM
        end=min(first+ACCUM,len(train))
        count=min(end*BATCH,len(train.sampler))-first*BATCH
        with torch.autocast('cuda',dtype=torch.float16):
            a,c=net(x,g)
            loss=masked_loss(a,y)+masked_loss(c,z)
        scaler.scale(loss*len(x)/count).backward()
        if i+1==end:
            scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(net.parameters(),1.)
            old=scaler.get_scale();scaler.step(opt);scaler.update()
            if scaler.get_scale()>=old:schedule.step()
            opt.zero_grad(set_to_none=True)
    val=predictions(net,'val')
    af,cf=head_scores(val)
    selection=(af+cf)/2 if cf is not None else af
    history.append(dict(epoch=epoch+1,action_f1=af,contact_macro_f1=cf,selection=selection))
    if selection>best:
        best=selection
        atomic_save({'format':'v2_contact_1','model':net.state_dict(),'epoch':epoch+1,'seen_contact':sorted(seen_contact),'base':BASE,'contact_labels':CONTACT},OUT/'best.pt')
    (OUT/'history.json').write_text(json.dumps(history,indent=2))
    print(history[-1])

### Chọn ngưỡng và đo lỗi

Dùng validation để chọn ngưỡng nghi vấn theo F1 hành động. Với cửa sổ hành động vượt ngưỡng, nhánh tiếp xúc có thể nâng mức cảnh báo khi cặp người đủ rõ và điểm chạm đủ cao. Đo số báo nhầm **gồm cả nghi ngờ**, số cảnh báo đánh nhau, số bỏ sót và số nguồn. Kết quả theo cửa sổ gắn nhãn. Khi thiếu dữ liệu tiếp xúc để hiệu chỉnh, demo chỉ xuất mức bình thường hoặc nghi ngờ.

In [ ]:
def decide(pa,pc,quality,policy):
    if pa<policy['action_threshold']: return 'normal'
    if (policy['contact_ready'] and quality and int(np.argmax(pc))==1
        and pc[1]>=policy['touch_threshold']): return 'fight'
    return 'suspected'

def calibrate(val,seen):
    grid=np.linspace(.05,.95,91)
    act=val[val.y>=0]
    at=max(grid,key=lambda t:(f1_score(act.y,act.p_action>=t,zero_division=0),-abs(t-.5)))
    observable=val[(val.z>=0)&(val.pose_ok==True)]
    ready=set(seen)=={0,1,2} and set(observable.z)=={0,1,2}
    ct=1.
    if ready:
        probs=observable[['p_no_touch','p_touch','p_unclear']].to_numpy()
        ct=max(grid,key=lambda t:f1_score(observable.z==1,(probs.argmax(1)==1)&(probs[:,1]>=t),zero_division=0))
    return dict(action_threshold=float(at),touch_threshold=float(ct),contact_ready=ready)

def report(df,policy,name):
    probs=df[['p_no_touch','p_touch','p_unclear']].to_numpy()
    df['decision']=[decide(a,c,q,policy) for a,c,q in zip(df.p_action,probs,df.pose_ok)]
    df.to_csv(OUT/f'{name}_predictions.csv',index=False)
    labeled=df[df.z>=0]
    if len(labeled):
        pred=labeled[['p_no_touch','p_touch','p_unclear']].to_numpy().argmax(1)
        print('Contact rows=true, cols=pred',CONTACT,confusion_matrix(labeled.z,pred,labels=[0,1,2]),sep='\n')
    table=[]
    for scenario,part in df.groupby('scenario'):
        neg=part[part.y==0];pos=part[part.y==1]
        table.append(dict(scenario=scenario,sources=part.source.nunique(),windows=len(part),
            normal=len(neg),strikes=len(pos),false_logs=int((neg.decision!='normal').sum()),
            false_fight=int((neg.decision=='fight').sum()),missed=int((pos.decision=='normal').sum()),
            review_strikes=int((pos.decision=='suspected').sum()),fight_strikes=int((pos.decision=='fight').sum())))
    result=pd.DataFrame(table)
    result['false_log_rate']=result.false_logs/result.normal.replace(0,np.nan)
    result['strike_recall_including_review']=1-result.missed/result.strikes.replace(0,np.nan)
    result.to_csv(OUT/f'{name}_summary.csv',index=False);display(result)

ck=torch.load(OUT/'best.pt',map_location='cpu',weights_only=True)
net.load_state_dict(ck['model'])
val=predictions(net,'val')
policy=calibrate(val,ck['seen_contact'])
(OUT/'policy.json').write_text(json.dumps(policy,indent=2))
print(policy)
report(val,policy,'val')
report(predictions(net,'test'),policy,'test')

### Thử clip mới hoặc các video KO

Điền đường dẫn vào KO_FILES. Dùng cùng checkpoint, phép chuẩn hóa, pose và ngưỡng. Các cửa sổ chồng lấn có thể tạo nhiều dòng cho cùng một sự kiện. Cột start_s/end_s giúp mở lại đúng đoạn. Một cửa sổ có thể phát cảnh báo cho cú đánh đơn. Báo cáo validation/test ở trên dùng cửa sổ có nhãn, còn phép thử này quét toàn clip.

In [ ]:
@torch.inference_mode()
def scan_video(path,step_s=.2):
    net.eval()
    path=str(Path(path).resolve())
    n,fps=info(path);duration=n/fps;digest=file_hash(path)
    rows=[]
    for length in sorted({min(w,duration) for w in SCALES}):
        last=duration-length
        starts=np.unique(np.append(np.arange(0,last,step_s),last))
        for start in starts:
            x,g,_=read_window(path,start,length,digest)
            a,c=net(x[None].to(device),g[None].to(device))
            pa=a.softmax(-1)[0,1].item();pc=c.softmax(-1)[0].cpu().numpy()
            ok=pose_quality(g[None]).item()
            rows.append(dict(path=path,start_s=start,end_s=start+length,p_action=pa,p_touch=pc[1],
                             p_unclear=pc[2],pose_ok=ok,decision=decide(pa,pc,ok,policy)))
    return pd.DataFrame(rows)
ko_dirs=[DATA_ROOT, MY_DRIVE/'train_v2_ko']
KO_FILES=[next(d/name for d in ko_dirs if (d/name).is_file())
          for name in ['KO_1s.mp4','KO3_2s.mp4','KO4_dai.mp4'] if any((d/name).is_file() for d in ko_dirs)]
if KO_FILES:
    ko=pd.concat([scan_video(p) for p in KO_FILES],ignore_index=True)
    ko.to_csv(OUT/'ko_windows.csv',index=False);display(ko)
print('Checkpoint:',OUT/'best.pt')